# stETH vs. ETH: 1-Minute Technical-Analysis Basis Arbitrage

Same pair as `steth_eth_trend_arb.ipynb` (Lido's stETH vs. ETH), tested with 1-minute intraday data
and classic technical-analysis signal constructs instead of daily rolling-regression slopes.

**Trend lines, made tradeable without lookahead bias**: a literal "connect the swing highs/lows"
trendline (Section 3, visual only) requires future bars to confirm a pivot, so it can't be used
directly as a live signal. The tradeable trend signal is a **rolling channel breakout** (price vs.
the rolling N-minute high/low on ETH) — fully causal, and a standard trend-following construct in
its own right (the Donchian channel breakout method).

**Price deviation**: the same stETH/ETH basis z-score construct as the daily notebook, with a much
shorter, intraday-appropriate lookback window.

**Hypothesis carried over from the daily notebook**: the 2022 stETH depeg happened because forced
deleveraging (leveraged staking-loop liquidations) hit during a crash, not a calm market — so ETH
breaking down through its rolling channel low should coincide with a wider stETH discount more than
calm, range-bound trading does.

**Data**: OKX's public API, 1-minute candles, paginated back to build several days of history.

> ⚠️ Research only, not investment advice. No transaction costs, staking/unstaking queue delays, or
> execution slippage modeled. At 1-minute resolution, trade count will be far higher than the daily
> version — Section 8's cost check matters even more here.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection (1-Minute, Paginated)


In [ ]:
OKX_BASE = "https://www.okx.com/api/v5/market"
TARGET_MINUTES = 7200  # ~5 days


def fetch_okx_minutes(inst_id, target_minutes=TARGET_MINUTES, bar="1m"):
    all_rows = []

    r = requests.get(f"{OKX_BASE}/candles", params={"instId": inst_id, "bar": bar, "limit": 100}, timeout=20)
    r.raise_for_status()
    data = r.json()["data"]
    all_rows.extend(data)
    after = min(int(row[0]) for row in data)

    while len(all_rows) < target_minutes:
        r = requests.get(
            f"{OKX_BASE}/history-candles",
            params={"instId": inst_id, "bar": bar, "limit": 100, "after": after},
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        after = min(int(row[0]) for row in data)
        time.sleep(0.2)

    cols = ["ts", "open", "high", "low", "close", "vol", "volCcy", "volCcyQuote", "confirm"]
    df = pd.DataFrame(all_rows, columns=cols)
    df = df[df["confirm"] == "1"]
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="ms")
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "vol"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "vol"]]


print("Fetching STETH-ETH 1-minute history...")
steth_eth = fetch_okx_minutes("STETH-ETH")
print("Fetching ETH-USDT 1-minute history...")
eth_usdt = fetch_okx_minutes("ETH-USDT")

print(f"\nSTETH-ETH: {steth_eth.index.min()} to {steth_eth.index.max()}, {len(steth_eth)} bars")
print(f"ETH-USDT:  {eth_usdt.index.min()} to {eth_usdt.index.max()}, {len(eth_usdt)} bars")


## 2. Basis Construction


In [ ]:
df = pd.DataFrame({
    "steth_eth_rate": steth_eth["close"],
    "eth_usdt": eth_usdt["close"],
}).dropna()

df["basis_pct"] = df["steth_eth_rate"] - 1

print(f"Overlapping sample: {df.index.min()} to {df.index.max()}, {len(df)} minutes")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["eth_usdt"], color="tab:blue")
axes[0].set_title("ETH-USDT Price — 1-Minute")

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green", linewidth=0.6)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("stETH/ETH Basis: (STETH-ETH rate - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Lines: Geometric (Visual) and Channel Breakout (Tradeable)


In [ ]:
def find_pivots(price, window=20):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=4):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).total_seconds().values / 60.0
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


recent_window = df.iloc[-2000:]
swing_highs, swing_lows = find_pivots(recent_window["eth_usdt"], window=20)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(recent_window.index, recent_window["eth_usdt"], color="tab:blue", label="ETH-USDT close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=20, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=20, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, recent_window.index[-1], freq="min")
        minutes_elapsed = (x_line - x0).total_seconds() / 60.0
        y_line = intercept + slope * minutes_elapsed
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("ETH (Recent Window): Swing-High / Swing-Low Trend Lines (Visual Only)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
CHANNEL_WINDOW = 30  # minutes

df["channel_high"] = df["eth_usdt"].rolling(CHANNEL_WINDOW).max().shift(1)
df["channel_low"] = df["eth_usdt"].rolling(CHANNEL_WINDOW).min().shift(1)

df["breakout_up"] = df["eth_usdt"] > df["channel_high"]
df["breakout_down"] = df["eth_usdt"] < df["channel_low"]
df["in_breakout"] = df["breakout_up"] | df["breakout_down"]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(recent_window.index, recent_window["eth_usdt"], color="tab:blue", linewidth=0.8, label="ETH-USDT close")
ax.plot(recent_window.index, df.loc[recent_window.index, "channel_high"], color="red", linewidth=0.8,
        linestyle="--", label=f"{CHANNEL_WINDOW}min rolling high")
ax.plot(recent_window.index, df.loc[recent_window.index, "channel_low"], color="green", linewidth=0.8,
        linestyle="--", label=f"{CHANNEL_WINDOW}min rolling low")
breakout_pts = recent_window.index[df.loc[recent_window.index, "in_breakout"]]
ax.scatter(breakout_pts, recent_window.loc[breakout_pts, "eth_usdt"], color="black", s=8, zorder=3,
           label="breakout bar")
ax.set_title(f"ETH (Recent Window): {CHANNEL_WINDOW}-Minute Channel Breakout Signal (Tradeable)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Breakout-up: {df['breakout_up'].sum()} bars, breakout-down: {df['breakout_down'].sum()} bars, "
      f"total breakout: {df['in_breakout'].mean()*100:.1f}% of the sample")


## 4. Does an ETH Channel Breakout Predict the stETH Basis?

The forced-deleveraging hypothesis specifically predicts **downside** breakouts should coincide with
a wider discount more than upside breakouts or calm periods — checked separately below, not just
breakout-vs-calm overall.


In [ ]:
def zscore(s, window=60):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

Z_WINDOW = 60
df["basis_z"] = zscore(df["basis_pct"], window=Z_WINDOW)

down_basis = df.loc[df["breakout_down"], "basis_z"].dropna()
up_basis = df.loc[df["breakout_up"], "basis_z"].dropna()
calm_basis = df.loc[~df["in_breakout"], "basis_z"].dropna()

print(f"Mean basis_z during DOWN breakouts: {down_basis.mean():.3f} (n={len(down_basis)})")
print(f"Mean basis_z during UP breakouts:   {up_basis.mean():.3f} (n={len(up_basis)})")
print(f"Mean basis_z outside breakouts:     {calm_basis.mean():.3f} (n={len(calm_basis)})")
print("Forced-deleveraging hypothesis predicts: down-breakout mean should be the most negative "
      "of the three (widest discount).")

fig, ax = plt.subplots()
ax.hist(calm_basis, bins=40, alpha=0.4, density=True, label="calm")
ax.hist(up_basis, bins=40, alpha=0.4, density=True, label="up breakout")
ax.hist(down_basis, bins=40, alpha=0.4, density=True, label="down breakout")
ax.set_title("stETH Basis Z-Score Distribution by ETH Regime")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Mean-Reversion Arbitrage Signal (on the Basis)


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)
print(f"Pure MR: {(position_pure_mr.diff().abs() > 0).sum()} trades over {len(df)} minutes")


## 6. Combining Both Ideas: Channel-Breakout-Filtered Arbitrage


In [ ]:
position_breakout_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=df["in_breakout"])

print(f"Minutes blocked by breakout filter: {df['in_breakout'].sum()} / {len(df)} "
      f"({df['in_breakout'].mean()*100:.1f}%)")
print(f"Breakout-filtered: {(position_breakout_filtered.diff().abs() > 0).sum()} trades")


In [ ]:
MINUTES_PER_YEAR = 365 * 24 * 60


def sharpe(returns, periods_per_year=MINUTES_PER_YEAR):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "breakout_filtered_mean_reversion": position_breakout_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Breakout-Filtered stETH/ETH Basis Arbitrage (1-Minute)")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(MINUTES_PER_YEAR),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "breakout_filtered_mean_reversion": (position_breakout_filtered.diff().abs() > 0).sum(),
    }),
})
summary


> ⚠️ **Read this before trusting the Sharpe numbers above**: annualizing a backtest this short by
> multiplying by `sqrt(minutes_per_year)` amplifies noise enormously. Treat `annualized_sharpe` as a
> diagnostic for comparing the two strategies *against each other on this same sample*, not a
> credible estimate of real annual risk-adjusted return. `total_return` (un-annualized, over the
> actual days fetched) is the more honest number to anchor on.


## 7. Market-Neutrality Check


In [ ]:
eth_ret = np.log(df["eth_usdt"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_eth": [strategies["pure_mean_reversion"].corr(eth_ret)],
    "breakout_filtered_vs_eth": [strategies["breakout_filtered_mean_reversion"].corr(eth_ret)],
}, index=["correlation_with_eth_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways

- **Check Section 4's three-way split, not just breakout-vs-calm**: the forced-deleveraging
  hypothesis makes a specific directional prediction (down-breakouts should show the widest
  discount, i.e. most negative basis_z) — a result where up and down breakouts look similar would
  argue against the specific mechanism even if breakouts overall differ from calm periods.
  Remember the daily-notebook sample also didn't show this (OKX's stETH-ETH data starts after the
  2022 depeg) — this ~5-day window is even less likely to contain a real stress event.
- **Trade frequency drives Section 8's result more than signal quality does**: compare `n_trades`
  here (over ~5 days) to the daily notebook's 48-76 trades (over ~3 years).
- **Sample size is short**: ~5 days of 1-minute bars. Re-run with a larger `TARGET_MINUTES` for a
  longer window.
- **Robustness**: validate out-of-sample and account for realistic costs (Section 8) before treating
  any of this as an actual edge.
